In [1]:

## CHANGE THIS
# run `ls /dev | grep ttyACM` to find the port
# flatpak run com.obsproject.Studio then `v4l2-ctl --list-devices` to find the camera index

follower_port = "/dev/ttyACM2"
leader_port = "/dev/ttyACM3"
camera_index = 4 # flatpak run obs studio. 

In [2]:

import os
import sys
parent_of_parent = os.path.dirname(os.path.dirname(os.getcwd()))
sys.path.append(parent_of_parent)

In [3]:

from lerobot.common.robot_devices.robots.configs import KochRobotConfig
from lerobot.common.robot_devices.robots.manipulator import ManipulatorRobot

from lerobot.common.robot_devices.motors.configs import DynamixelMotorsBusConfig
from lerobot.common.robot_devices.motors.dynamixel import DynamixelMotorsBus

from lerobot.common.robot_devices.cameras.configs import OpenCVCameraConfig
from lerobot.common.robot_devices.cameras.opencv import OpenCVCamera

In [4]:


leader_config = DynamixelMotorsBusConfig(
    port=leader_port,
    motors={
        # name: (index, model)
        "shoulder_pan": (1, "xl330-m077"),
        "shoulder_lift": (2, "xl330-m077"),
        "elbow_flex": (3, "xl330-m077"),
        "wrist_flex": (4, "xl330-m077"),
        "wrist_roll": (5, "xl330-m077"),
        "gripper": (6, "xl330-m077"),
    },
)

follower_config = DynamixelMotorsBusConfig(
    port=follower_port,
    motors={
        # name: (index, model)
        "shoulder_pan": (1, "xl430-w250"),
        "shoulder_lift": (2, "xl430-w250"),
        "elbow_flex": (3, "xl330-m288"),
        "wrist_flex": (4, "xl330-m288"),
        "wrist_roll": (5, "xl330-m288"),
        "gripper": (6, "xl330-m288"),
    },
)

leader_arm = DynamixelMotorsBus(leader_config)
follower_arm = DynamixelMotorsBus(follower_config)

In [5]:

robot_config = KochRobotConfig(
    leader_arms={"main": leader_config},
    follower_arms={"main": follower_config},
    calibration_dir=".cache/calibration/koch",
    cameras={
        # "laptop": OpenCVCameraConfig(0, fps=30, width=640, height=480),
        "phone": OpenCVCameraConfig(camera_index=camera_index, fps=30, width=640, height=480),
    },
    # cameras={},
)
robot = ManipulatorRobot(robot_config)
robot.connect()

Connecting main follower arm.
Connecting main leader arm.
Activating torque on main follower arm.


[ WARN:0@1.191] global cap_gstreamer.cpp:2839 handleMessage OpenCV | GStreamer warning: Embedded video playback halted; module typefind reported: Could not determine type of stream.
[ WARN:0@1.192] global cap_gstreamer.cpp:1698 open OpenCV | GStreamer warning: unable to start pipeline
[ WARN:0@1.192] global cap_gstreamer.cpp:1173 isPipelinePlaying OpenCV | GStreamer warning: GStreamer: pipeline have not been created
[ WARN:0@2.358] global cap_gstreamer.cpp:2839 handleMessage OpenCV | GStreamer warning: Embedded video playback halted; module typefind reported: Could not determine type of stream.
[ WARN:0@2.358] global cap_gstreamer.cpp:1698 open OpenCV | GStreamer warning: unable to start pipeline
[ WARN:0@2.358] global cap_gstreamer.cpp:1173 isPipelinePlaying OpenCV | GStreamer warning: GStreamer: pipeline have not been created


In [6]:
import time
import torch
from lerobot.common.policies.act.modeling_act import ACTPolicy
from lerobot.common.policies.diffusion.modeling_diffusion import DiffusionPolicy
from lerobot.scripts.control_robot import busy_wait

inference_time_s = 20 # 20s for rollout
fps = 30
device = "cpu"

# ckpt_path = "ellen2imagine/act_koch_test"
# ckpt_path = "pmarsella/act_v2"
# policy = ACTPolicy.from_pretrained(ckpt_path)
ckpt_path = "pmarsella/diffusion_pusht"
policy = DiffusionPolicy.from_pretrained(ckpt_path)
policy.to(device)

/opt/miniconda3/envs/lerobot/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DiffusionPolicy(
  (normalize_inputs): Normalize(
    (buffer_observation_state): ParameterDict(
        (max): Parameter containing: [torch.FloatTensor of size 6]
        (min): Parameter containing: [torch.FloatTensor of size 6]
    )
    (buffer_observation_images_phone): ParameterDict(
        (mean): Parameter containing: [torch.FloatTensor of size 3x1x1]
        (std): Parameter containing: [torch.FloatTensor of size 3x1x1]
    )
  )
  (normalize_targets): Normalize(
    (buffer_action): ParameterDict(
        (max): Parameter containing: [torch.FloatTensor of size 6]
        (min): Parameter containing: [torch.FloatTensor of size 6]
    )
  )
  (unnormalize_outputs): Unnormalize(
    (buffer_action): ParameterDict(
        (max): Parameter containing: [torch.FloatTensor of size 6]
        (min): Parameter containing: [torch.FloatTensor of size 6]
    )
  )
  (diffusion): DiffusionModel(
    (rgb_encoder): DiffusionRgbEncoder(
      (center_crop): CenterCrop(size=(84, 84))
      

In [7]:
# move to starting position
import numpy as np
target = np.array([ 69.1699,  83.4082, 122.9590,  47.8125, 184.7461,  -7.0312])
robot.follower_arms["main"].write("Goal_Position", target)

In [8]:

s_total = time.perf_counter()
frames = []
log_data = []

for ts in range(inference_time_s * fps):
    start_time = time.perf_counter()

    # Read the follower state and access the frames from the cameras
    observation = robot.capture_observation()

    # Convert to pytorch format: channel first and float32 in [0,1]
    # with batch dimension
    for name in observation:
        if "image" in name:
            observation[name] = observation[name].type(torch.float32) / 255
            observation[name] = observation[name].permute(2, 0, 1).contiguous()
            frames.append(observation[name])
        observation[name] = observation[name].unsqueeze(0)
        observation[name] = observation[name].to(device)

    # Compute the next action with the policy
    # based on the current observation
    s_time = time.perf_counter()
    action = policy.select_action(observation)
    e_time = time.perf_counter()
    frame_time = e_time - s_time
    print(f"Time taken: {e_time - s_time} seconds")
    # Remove batch dimension
    action = action.squeeze(0)
    # Move to cpu, if not already the case
    action = action.to("cpu")
    # Order the robot to move
    robot.send_action(action)

    log_entry = {
        "frame": ts,
        "frame_time": frame_time,
        "observation_state": observation["observation.state"].cpu().numpy().tolist(),
        "action": action.cpu().numpy().tolist()
    }
    log_data.append(log_entry)

    dt_s = time.perf_counter() - start_time
    busy_wait(1 / fps - dt_s)

total_time = time.perf_counter() - s_total
print(f"Total time taken: {total_time} seconds")

Time taken: 32.78031434500008 seconds
Time taken: 0.0007485309906769544 seconds
Time taken: 0.0008175530092557892 seconds
Time taken: 0.002452765009365976 seconds
Time taken: 0.0015751900064060465 seconds
Time taken: 0.002147424005670473 seconds
Time taken: 0.0012642939982470125 seconds
Time taken: 0.0015134510031202808 seconds
Time taken: 39.80297073601105 seconds
Time taken: 0.0008116370008792728 seconds
Time taken: 0.0006805060111219063 seconds
Time taken: 0.0007222279964480549 seconds
Time taken: 0.0007873650029068813 seconds
Time taken: 0.0012995860015507787 seconds
Time taken: 0.0010281180002493784 seconds
Time taken: 0.0007356159912887961 seconds
Time taken: 35.02626439099549 seconds
Time taken: 0.0011752669961424544 seconds
Time taken: 0.0007980340014910325 seconds
Time taken: 0.0009607749962015077 seconds
Time taken: 0.0013207009906182066 seconds
Time taken: 0.0012529079976957291 seconds
Time taken: 0.0012990779941901565 seconds
Time taken: 0.0009104989876504987 seconds
Time t

ConnectionError: Read failed due to communication error on port /dev/ttyACM2 for group_key Present_Position_shoulder_pan_shoulder_lift_elbow_flex_wrist_flex_wrist_roll_gripper: [TxRxResult] There is no status packet!

In [9]:
i = "diffusion" # which eval episode

In [10]:

import json
import datetime

log_path = f"outputs/eval/rollout_{i}.json"
os.makedirs(os.path.dirname(log_path), exist_ok=True)

log_summary = {
    "total_time": total_time,
    "fps": fps,
    "intended_duration": inference_time_s,
    "frames_count": len(frames),
    "policy": ckpt_path,
    "device": device,
    "frames_data": log_data
}

with open(log_path, "w") as f:
    json.dump(log_summary, f, indent=2)

print(f"Saved log data to {log_path}")

NameError: name 'total_time' is not defined

In [11]:

# save last frame of phone camera. we do 5 total evals
# last_observation = robot.capture_observation()
# last_frame = last_observation["observation.images.phone"]

last_frame = frames[-1]
import os
os.makedirs("outputs/eval", exist_ok=True)

import cv2
import numpy as np

# Convert from CHW (3, 480, 640) to HWC (480, 640, 3) format
last_frame_np = last_frame.permute(1, 2, 0).numpy()
# Scale back to 0-255 range and convert to uint8
last_frame_np = (last_frame_np * 255).astype(np.uint8)
# Convert from RGB to BGR for OpenCV
last_frame_bgr = cv2.cvtColor(last_frame_np, cv2.COLOR_RGB2BGR)

cv2.imwrite(f"outputs/eval/eval{i}.png", last_frame_bgr)
# cv2.imshow("eval", last_frame_bgr)
# cv2.waitKey(0)
# cv2.destroyAllWindows()
print(f"Saved eval{i}.png")

Saved evaldiffusion.png


In [12]:

video_path = f"outputs/eval/rollout_{i}.mp4"
# Get dimensions from the first frame
height, width = frames[0].shape[1], frames[0].shape[2]  # Frames are in CHW format

# Try using a different codec that's more widely supported
# 'avc1' is H.264 which has better compatibility than 'mp4v'
fourcc = cv2.VideoWriter_fourcc(*'avc1')  # Alternative: 'XVID' for .avi format

video_writer = cv2.VideoWriter(video_path, fourcc, fps, (width, height))

for frame in frames:
    # Convert from CHW to HWC format and from tensor to numpy
    frame_np = frame.permute(1, 2, 0).numpy()
    # Scale back to 0-255 range and convert to uint8
    frame_np = (frame_np * 255).astype(np.uint8)
    # Convert from RGB to BGR for OpenCV
    frame_bgr = cv2.cvtColor(frame_np, cv2.COLOR_RGB2BGR)
    video_writer.write(frame_bgr)

video_writer.release()
print(f"Saved video to {video_path}")

Saved video to outputs/eval/rollout_diffusion.mp4
